In [1]:

import osmnx as ox
import pandas as pd

In [2]:

# Target area
place_name = "Nancy, France"

# Tags to extract for the LLM dataset
tags = {
    'amenity': True,         # Restaurants, cafes, schools, etc.
    'historic': True,        # Monuments and heritage sites
    'tourism': ['museum', 'hotel', 'attraction'],
    'leisure': 'park'        # Green spaces
}

print(f"Retrieving geospatial features for {place_name}...")

Retrieving geospatial features for Nancy, France...


In [3]:

pois = ox.features_from_place(place_name, tags)

pois_points = pois[pois.geometry.type == 'Point'].copy()

In [4]:

pois_points['latitude'] = pois_points.geometry.y
pois_points['longitude'] = pois_points.geometry.x

In [5]:
# Gather necessary or possibly useful information
data_columns = ['name', 'amenity','latitude', 'longitude']
df_nancy = pois_points[[col for col in data_columns if col in pois_points.columns]]

In [6]:

# Preview the first few rows
df_nancy.head(10)

name         amenity  \
element id                                                            
node    262840256                               NaN        post_box   
        309525299                               NaN       recycling   
        313719091                 Place Commanderie  bicycle_rental   
        313719092             Commanderie - Chalnot  bicycle_rental   
        313719095                   Gare Saint-Léon  bicycle_rental   
        313734278  Saint-Sébastien - Saint-Thiébaut  bicycle_rental   
        313734964                    Marché Central  bicycle_rental   
        313735272                    Place Dombasle  bicycle_rental   
        313736384              Place de la Carrière  bicycle_rental   
        313736860      Cours Léopold - Saint-Michel  bicycle_rental   

                    latitude  longitude  
element id                               
node    262840256  48.687206   6.162858  
        309525299  48.688398   6.160095  
        313719091  48.685852   6.167446  
        313719092  48.686859   6.172236  
        313719095  48.689593   6.172298  
        313734278  48.688691   6.179350  
        313734964  48.689170   6.183843  
        313735272  48.691851   6.178190  
        313736384  48.694573   6.182192  
        313736860  48.695718   6.176774

In [7]:

df_nancy.info()

<class 'pandas.DataFrame'>
MultiIndex: 5447 entries, ('node', np.int64(262840256)) to ('node', np.int64(13667270477))
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       1198 non-null   str    
 1   amenity    5304 non-null   str    
 2   latitude   5447 non-null   float64
 3   longitude  5447 non-null   float64
dtypes: float64(2), str(2)
memory usage: 259.0+ KB


In [8]:
df_nancy = df_nancy.dropna()

In [9]:
df_nancy.info()

<class 'pandas.DataFrame'>
MultiIndex: 1111 entries, ('node', np.int64(313719091)) to ('node', np.int64(13667270477))
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       1111 non-null   str    
 1   amenity    1111 non-null   str    
 2   latitude   1111 non-null   float64
 3   longitude  1111 non-null   float64
dtypes: float64(2), str(2)
memory usage: 110.8+ KB


Sentence Generation process:
iterate through the dataframe.
for every other location in the dataframe generate a direction
Create result sentences in format: Name2 is DIRECTION of Name1

In [10]:
df_nancy['name'].iloc[0]

'Place Commanderie'

In [11]:
sentence_list = list()
for i in range(len(df_nancy)):
    for j in range(len(df_nancy)):
        if df_nancy['name'].iloc[i] != df_nancy['name'].iloc[j]:
            lat = df_nancy['latitude'].iloc[j] - df_nancy['latitude'].iloc[i]
            lon = df_nancy['longitude'].iloc[j] - df_nancy['longitude'].iloc[i]
            cardinal = ""
            if abs(lat) >= abs(lon):
                if lat > 0:
                    cardinal = 'NORTH'
                else:
                    cardinal = 'SOUTH'
            else:
                if lon > 0:
                    cardinal = 'EAST'
                else:
                    cardinal = 'WEST'
            sentence_list.append(f'{df_nancy['name'].iloc[j]} is {cardinal} of {df_nancy['name'].iloc[i]}')

In [16]:
sentence_list[100:200]

['Les Oliviers is EAST of Place Commanderie',
 'Akoya Sushi is NORTH of Place Commanderie',
 'Pharmacie de la Croix is NORTH of Place Commanderie',
 'CIC is NORTH of Place Commanderie',
 'Crédit Agricole is NORTH of Place Commanderie',
 'Little Faubourg is NORTH of Place Commanderie',
 'Pharmacie de la Craffe is NORTH of Place Commanderie',
 'CIC is NORTH of Place Commanderie',
 'Crédit Agricole is NORTH of Place Commanderie',
 'Crédit Agricole is NORTH of Place Commanderie',
 '3 Maisons Kebab is NORTH of Place Commanderie',
 'Brasserie de la Croix is NORTH of Place Commanderie',
 'La Citadelle is NORTH of Place Commanderie',
 'BNP Paribas is NORTH of Place Commanderie',
 'Banque Populaire is NORTH of Place Commanderie',
 'BNP Paribas is NORTH of Place Commanderie',
 'Madame is NORTH of Place Commanderie',
 'Hops Bar is NORTH of Place Commanderie',
 'La Brûlerie is NORTH of Place Commanderie',
 'The Reddington Bar is NORTH of Place Commanderie',
 'Le Pacha du Bosphore is NORTH of Place

In [17]:
with open('directionSentences.txt', 'w') as f:
    for sentence in sentence_list:
        f.write(f"{sentence}\n")